# 📚 LAB 1: Nearest Neighbor (NN) - Thuật Toán Tham Lam

## Khái Niệm
**Nearest Neighbor** là thuật toán **tham lam (Greedy)**:
- Luôn chọn **khách gần nhất** chưa được thăm
- **Không nhìn xa** → Cục bộ tối ưu, **không toàn cục**
- **Nhanh:** O(n²) time
- **Cơ bản:** Đơn giản, dễ hiểu

## Pseudo-code
```
current = depot
route = []
while có khách chưa thăm:
    next = khách gần nhất từ current
    route.append(next)
    current = next
return route
```

## Ứng Dụng Thực Tế
- ✅ GPS navigation (chỉ đường nhanh)
- ✅ Delivery routing (tham khảo)
- ❌ Tối ưu hóa chi phí (kém, cần heuristic cao cấp)

In [ ]:
import csv
import math
import random
from pathlib import Path
import matplotlib.pyplot as plt

DATA_FILE = Path("dataset_dvrp_380.csv")
DEPOT = (50, 50)
CAPACITY = 25

def load_data(path):
    with path.open("r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["id"], r["demand"], r["appear_time"] = int(r["id"]), int(r["demand"]), int(r["appear_time"])
        r["x"], r["y"] = float(r["x"]), float(r["y"])
    return rows

def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def tour_length(route):
    if not route: return 0.0
    d = dist(DEPOT, (route[0]["x"], route[0]["y"]))
    d += sum(dist((route[i]["x"], route[i]["y"]), (route[i+1]["x"], route[i+1]["y"])) for i in range(len(route)-1))
    d += dist((route[-1]["x"], route[-1]["y"]), DEPOT)
    return d

customers = load_data(DATA_FILE)
print(f"✅ Loaded {len(customers)} customers")
print(f"   Depot: {DEPOT}, Capacity: {CAPACITY}")

## Bước 2: Nearest Neighbor Algorithm

In [ ]:
def nn_order(active):
    """🎯 Nearest Neighbor: Chọn khách gần nhất"""
    if not active: return []
    unvisited = {c["id"]: c for c in active}
    route, cur_pos, cur_id = [], DEPOT, 0
    
    while unvisited:
        # Chọn khách gần nhất
        bid = min(unvisited.keys(), key=lambda cid: dist(cur_pos, (unvisited[cid]["x"], unvisited[cid]["y"])))
        route.append(unvisited[bid])
        cur_pos, cur_id = (unvisited[bid]["x"], unvisited[bid]["y"]), bid
        del unvisited[bid]
    
    return route

def simulate_nn(customers):
    """Mô phỏng NN: Khách xuất hiện → Dispatch theo NN"""
    backlog, served = [], set()
    dispatch_detail, total = [], 0.0
    
    for t in range(max(c["appear_time"] for c in customers) + 1):
        backlog += [c for c in customers if c["appear_time"] == t and c["id"] not in served]
        
        while backlog:
            trip, load = [], 0
            for c in nn_order(backlog):  # NN order
                if load + c["demand"] <= CAPACITY:
                    trip.append(c)
                    load += c["demand"]
            
            if not trip: break
            cost = tour_length(trip)
            total += cost
            served.update(c["id"] for c in trip)
            backlog = [c for c in backlog if c["id"] not in served]
            dispatch_detail.append({"time": t, "trip_ids": [c["id"] for c in trip], "cost": cost, "load": load})
    
    return dispatch_detail, total

# Chạy NN
nn_detail, nn_total = simulate_nn(customers)
nn_dispatches = len(nn_detail)
print(f"\n✅ NN RESULTS:")
print(f"   Total Cost: {nn_total:.2f}")
print(f"   Dispatches: {nn_dispatches}")
print(f"   Avg Cost/Dispatch: {nn_total/nn_dispatches:.2f}")

## Bước 3: Trực Quan Hóa

In [ ]:
def plot_nn_results(customers, nn_detail, nn_total):
    xs, ys = [c["x"] for c in customers], [c["y"] for c in customers]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Hình 1: Bản đồ + Lộ trình
    ax1.scatter(xs, ys, s=50, alpha=0.6, label="Customers")
    ax1.scatter([DEPOT[0]], [DEPOT[1]], marker="*", s=400, c="red", label="Depot")
    
    cust = {c["id"]: c for c in customers}
    colors = plt.cm.tab20(range(min(len(nn_detail), 20)))
    
    for i, d in enumerate(nn_detail[:20]):  # Vẽ 20 chuyến đầu
        if not d["trip_ids"]: continue
        pts = [DEPOT] + [(cust[cid]["x"], cust[cid]["y"]) for cid in d["trip_ids"]] + [DEPOT]
        for p1, p2 in zip(pts[:-1], pts[1:]):
            ax1.arrow(p1[0], p1[1], p2[0]-p1[0], p2[1]-p1[1], head_width=0.8, head_length=1.2, 
                     linewidth=1.0, color=colors[i % len(colors)], alpha=0.7)
    
    ax1.set_xlim(0, 102)
    ax1.set_ylim(0, 102)
    ax1.set_title(f"NN: {len(customers)} customers, {len(nn_detail)} dispatches, Cost={nn_total:.0f}", fontsize=12, weight="bold")
    ax1.grid(alpha=0.3)
    ax1.legend()
    
    # Hình 2: Chi phí theo dispatch
    costs = [d["cost"] for d in nn_detail]
    ax2.bar(range(len(costs)), costs, color="steelblue", alpha=0.7)
    ax2.set_xlabel("Dispatch #")
    ax2.set_ylabel("Cost (distance)")
    ax2.set_title(f"Cost Distribution (Min={min(costs):.0f}, Max={max(costs):.0f}, Avg={sum(costs)/len(costs):.0f})")
    ax2.grid(alpha=0.3, axis="y")
    
    plt.tight_layout()
    plt.show()

plot_nn_results(customers, nn_detail, nn_total)

## 📊 NOTE: So Sánh NN vs DVRP

### **Kết Quả Số Liệu:**

| Tiêu chí | NN | DVRP (ACS) | Winner | 
|----------|----|-----------|---------|
| **Chi phí tổng** | ~13,868 | ~13,528 | 🟢 **DVRP tốt 2.5%** |
| **Số chuyến** | 152 | 152 | Bằng |
| **Chi phí/chuyến** | 91.1 | 89.0 | 🟢 **DVRP tốt hơn** |
| **Thời gian tính** | < 1s | ~5s | 🟢 **NN nhanh** |
| **Chất lượng** | Tham lam | Tối ưu | 🟢 **DVRP tốt** |

### **Constraint & Đặc Điểm:**

| Khía cạnh | NN | DVRP |
|----------|----|---------|
| **Constraint** | Chỉ capacity | Capacity + Pheromone + Event |
| **Flexible** | Cứng nhắc (cục bộ) | Linh hoạt (toàn cục) |
| **Học hỏi từ quá khứ** | ❌ Không | ✅ Có (pheromone) |
| **Phù hợp** | ✅ Nhanh quyết định | ✅ Tối ưu chi phí |
| **Mục đích** | GPS, tham khảo | Logistics thực tế |

### **Kết Luận:**
- **NN:** ✅ **Nhanh**, ❌ **Kém tối ưu** → Dùng khi cần **tốc độ**
- **DVRP:** ❌ **Chậm hơn**, ✅ **Tối ưu 2-3%** → Dùng khi cần **chi phí thấp**
- **Khuyến nghị:** DVRP **tốt hơn cho logistics thực tế** vì tiết kiệm chi phí lâu dài